Almacenamiento en caché

- Permite que Spark conserve los datos en todos los cálculos y operaciones
- Podemos marcar un RDD como almacenado en caché usando persist() o cache()
- cache() es simplemente un sinónimo fr persist(MEMORY_ONLY)
- persist() puede usar memoria o disco o ambos

Valores posibles para el nivel de almacenamiento

Nivel de Almacenamiento

MEMORY_ONLY
- Almacena RDD como objetos Java deserializados en JVM. Si el RDD no cabe en la memoria, algunas particiones no se almacenarán en caché y se volverán a calcular sobre la marcha cada vez que se necesiten. Este es el nivel por defecto.

MEMORY_AND_DISK
- Almacena RDD como objetos Java deserializados en la JVM. Si el RDD no cabe en la memoria, almacena las particiones que no quepan en el disco y las lee desde allí cuando sea necesario.

DISK_ONLY
- Almacene las partciones RDD solo en el disco.

MEMORY_ONLY_2, MEMORY_AND_DISK_2, etc.
- Igual que los niveles anteriores, pero replica cada partición en dos nodos del clúster.

El nivel de almacenamiento a elegir depende de la situación

- Si los RDD caben en la memoria, use MEMORY_ONLY ya que es la opción más rápida para el rendimiento de ejecución
- DISK_ONLY no debe usarse a menos que sus cálculos sean costosos
- Utilice el almacenamiento replicado para una mejor tolerancia a fallas si puede ahorrar la memoria adicional necesaria. Esto evitará que se vuelvan a calcular las particiones perdidas para obtener la mejor disponibilidad


In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

sc=spark.sparkContext

In [8]:
rdd= sc.parallelize([item for item in range(10)])

In [9]:
from pyspark.storagelevel import StorageLevel

rdd.persist(StorageLevel.MEMORY_ONLY)


ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:297

In [10]:
#Si queremos cambiar el nivel de persistencia debemos usar unpersist
rdd.unpersist()

ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:297

In [11]:
rdd.persist(StorageLevel.DISK_ONLY)

ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:297

In [ ]:
rdd.unpersist()

ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:297

In [13]:
#Cache es un sinonimo de Memory only
rdd.cache()

ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:297

Particionado y mezcla de datos(shuffling)

Los RDD operan con datos no como una sola masa de datos, sino que administran y operan los datos en partciones repartidas por todo el clúster.

El número de partciones es importante

- Si la cantidad de particiones es demasiado pequeña, usaremos solo unas pocas CPU/núcleos en una gran cantidad de datos, por lo que tendremos un rendimiento más lento y dejaremos el clúster subutilizado
- Si la cantidad de particiones es demasiado grande, utilizará más recursos de los que realmente necesita y, en un entorno de múltiples procesos, podría estar provocando la falta de recursos para otros procesos que usted u otros miembros de su equipo ejecutan

Particionadores

Particionadores de Spark

- HashPartitioner
- RangePartitioner



HashPartitioner

- Es el particionador predeterminado en Spark

Formula:

partitionIndex= hask(key)%numPartitions

RangePartitioner

- Funciona dividiendo el RDD en rangos aproximadamente iguales
- Primero necesita límites razonables para las particiones basadas en el RDD
- Luego crea una función desde la clave K hasta el partitionIndex al que pertenece el elemento
- Finalmente, necesitamos reparticionar el RDD, basado en el RangePartitioner para distribuir los elementos del RDD correctamente según los rangos que determinamos

HashPartitioner

In [14]:
rdd= sc.parallelize(['x', 'y', 'z'])

In [15]:
hola= 'Hola'
hash(hola)

-5189199322048128668

In [17]:
num_particiones=6

In [ ]:
#indice = hash(item)%num_particiones

In [24]:
hash('x')%num_particiones

4

In [25]:
hash('y')%num_particiones

2

In [26]:
hash('z')%num_particiones

4

Mezcla de datos(shuffling)

El movimiento de datos necesario para el reparticionamiento se denomina shuffling

Broadcast variables

Son variables compartidas entre todos los ejecutores. Estas se crean una vez en el controlador y luego se leen sólo en los ejecutores.

In [27]:
rdd= sc.parallelize([item for item in (range(10))])

In [28]:
uno=1
br_uno= sc.broadcast(uno)

In [30]:
rdd1= rdd.map(lambda x: x + br_uno.value)
rdd1.collect()

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

In [ ]:
#Para eliminar los datos de la variable broadcast de la memoria de cache de todos los ejecutores
br_uno.unpersist()

In [33]:
rdd1= rdd.map(lambda x: x +br_uno.value)
rdd1.collect()

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

In [34]:
#Para destruir la variable broadcast eliminando de los ejecutores y del controlador haciendo innacesibles
br_uno.destroy()

In [ ]:
rdd1= rdd.map(lambda x: x +br_uno.value)
rdd1.take(5)

Acumuladores

Son variables compartidas entre ejecutores que normalmente se utilizan para agregar contadores a su programa Spark.

- sparkContext.accumulator() se usa para definir variables de acumulador
- La función add() se usa para agregar/actualizar un valor en el acumulador
- La propiedad value de la variable del acumulador se utiliza para recuperar el valor del acumulador

- add(1) → contar 
- add(x) → sumar el valor 

In [39]:
acumulador= sc.accumulator(0)

In [49]:
rdd=sc.parallelize([2,4,6,8,10])

In [51]:
rdd.foreach(lambda x: acumulador.add(x))
print(acumulador.value)

210


In [52]:
rdd1= sc.parallelize('Mi nombre es Danny y me encuentro super bien'.split(' '))

In [53]:
acumulador1= sc.accumulator(0)

In [58]:
rdd1.foreach(lambda x: acumulador1.add(1))

In [59]:
print(acumulador1.value)

18
